# 🧠 EpiConnectome Tutorial
## A Reproducible Pipeline for EEG Connectivity Analysis of Epileptic Seizures

**Brainhack School 2026 | Author: Mariya Duh | NCU Taiwan**

[![Python](https://img.shields.io/badge/Python-3.11-blue.svg)](https://www.python.org/)
[![MNE](https://img.shields.io/badge/MNE--Python-1.6+-green.svg)](https://mne.tools/)
[![Dataset](https://img.shields.io/badge/Dataset-PhysioNet%20Siena-orange.svg)](https://physionet.org/content/siena-scalp-eeg/1.0.0/)

---

This notebook walks through the **full EpiConnectome pipeline** step by step:

| Script | Description |
|--------|-------------|
| `01_siena_to_txt_converter.py` | Convert EDF seizure recordings → TXT segments |
| `02_channelbasedcode_Siena.py` | Preprocessing + wPLI connectivity analysis |
| `03_siena_feature_extraction.py` | EEG feature extraction (entropy, variance, skewness) |
| `04_siencehisto.py` | Connectivity matrix visualization |
| `05_dspm_connectivity.py` | Source-level analysis with dSPM + HCPMMP1 parcellation |
| `06_gatv2_classification.py` | GATv2 graph neural network seizure classification |

> ⚠️ **Data note:** Raw EEG data (13 GB) must be downloaded separately from PhysioNet.  
> See the [Dataset section](#dataset) below.


---
## 📦 Dataset

**Siena Scalp EEG Database (v1.0.0)** — PhysioNet

| Property | Value |
|----------|-------|
| Patients | 14 subjects |
| Seizures | 47 seizures |
| Channels | 29 scalp electrodes (10-20 system) |
| Sampling rate | 512 Hz |
| Format | EDF + seizure annotations |

Download with:
```bash
wget -r -N -c -np https://physionet.org/files/siena-scalp-eeg/1.0.0/
```
Or visit: https://physionet.org/content/siena-scalp-eeg/1.0.0/

**Citation:**
> Detti, P. (2020). Siena Scalp EEG Database (version 1.0.0). PhysioNet. https://doi.org/10.13026/5d4a-j060


---
## ⚙️ Setup & Installation

Run the cell below to verify your environment has all required packages.


In [ ]:
# Verify environment
import importlib, sys

required = [
    'mne', 'mne_connectivity', 'numpy', 'scipy',
    'matplotlib', 'pandas', 'networkx', 'sklearn',
    'torch', 'torch_geometric', 'openpyxl', 'seaborn'
]

print(f"Python: {sys.version.split()[0]}")
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'installed')
        print(f"  ✅ {pkg}: {version}")
    except ImportError:
        print(f"  ❌ {pkg}: NOT FOUND — run: pip install {pkg}")


---
## Step 1 — Convert EDF Files to TXT
### `01_siena_to_txt_converter.py`

**Purpose:** Extract seizure segments from raw EDF recordings and save them as TXT files for downstream processing.

**What it does:**
- Reads EDF files and their associated seizure annotation files (`.txt` / `.tsv`)
- Identifies seizure onset and offset times
- Extracts EEG segments with a configurable buffer around each seizure
- Saves each segment as a space-delimited TXT file with channel names as headers

**Key parameters:**
| Parameter | Value | Description |
|-----------|-------|-------------|
| Pre-seizure buffer | 30s | Context before seizure onset |
| Post-seizure buffer | 30s | Context after seizure offset |
| Channels | 29 (10-20) | All scalp electrodes |
| Output format | TXT (space-delimited) | One row per sample |


In [ ]:
# Example: What a converted TXT file looks like
import pandas as pd
import numpy as np

# Simulated output structure (real data requires download)
n_samples = 512 * 10  # 10 seconds at 512 Hz
channels = ['Fp1','Fp2','F7','F3','Fz','F4','F8',
            'T3','C3','Cz','C4','T4','T5','P3','Pz','P4']

demo_data = pd.DataFrame(
    np.random.randn(n_samples, len(channels)) * 50,  # ~50 µV amplitude
    columns=channels
)
print(f"Shape: {demo_data.shape}  ({n_samples} samples × {len(channels)} channels)")
print(f"Duration: {n_samples/512:.1f} seconds at 512 Hz")
print()
print(demo_data.head(3).round(2))


---
## Step 2 — Preprocessing + wPLI Connectivity
### `02_channelbasedcode_Siena.py`

**Purpose:** Full EEG preprocessing pipeline followed by weighted Phase Lag Index (wPLI) connectivity computation across three frequency bands.

### 2a. Preprocessing Steps

```
Raw TXT → Load → Bandpass filter (4–40 Hz) → Resample (512→1000 Hz)
       → Bad channel detection (LOF) → ICA artifact removal (infomax, 15 components)
       → Re-reference (average) → Epoch (10s windows)
```

### 2b. Connectivity Analysis

For each epoch, wPLI is computed between all channel pairs in:
- **Theta:** 6–8 Hz
- **Alpha:** 8–12 Hz  
- **Beta:** 12–30 Hz

### 2c. Graph Theory Metrics

| Metric | Description |
|--------|-------------|
| Global Efficiency | Information exchange efficiency |
| Average Clustering | Local cluster formation |
| Network Density | Fraction of active connections |
| Modularity | Community separation |
| Node Strength | Per-channel connection weight sum |


In [ ]:
# Conceptual demonstration of wPLI connectivity
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Simulate wPLI matrix (16x16 channels)
np.random.seed(42)
channels_16 = ['Fp1','Fp2','F7','F3','Fz','F4','F8',
               'T3','C3','Cz','C4','T4','T5','P3','Pz','P4']

# Simulate seizure-period connectivity (elevated, ~0.4-0.8)
wpli_matrix = np.random.uniform(0.4, 0.8, (16, 16))
np.fill_diagonal(wpli_matrix, 0)
wpli_matrix = (wpli_matrix + wpli_matrix.T) / 2  # symmetric

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(wpli_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(16))
ax.set_yticks(range(16))
ax.set_xticklabels(channels_16, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(channels_16, fontsize=8)
ax.set_title('Simulated wPLI Connectivity Matrix (Seizure Period — Beta Band)', fontsize=11)
plt.colorbar(im, ax=ax, label='wPLI')
plt.tight_layout()
plt.savefig('demo_wpli_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print("wPLI matrix shape:", wpli_matrix.shape)
print(f"Mean connectivity: {wpli_matrix[wpli_matrix>0].mean():.3f}")


---
## Step 3 — EEG Feature Extraction
### `03_siena_feature_extraction.py`

**Purpose:** Compute four EEG features per channel per frequency band that quantify signal complexity and distribution.

| Feature | Formula | Interpretation |
|---------|---------|---------------|
| **Sample Entropy** | SampEn(m, r, N) | Signal irregularity/complexity |
| **Spectral Entropy** | -Σ p(f) log p(f) | Frequency distribution uniformity |
| **Variance** | σ² | Signal amplitude variability |
| **Skewness** | μ₃/σ³ | Asymmetry of amplitude distribution |

**Output:** Feature heatmaps per subject + aggregated Excel file with all features across channels and bands.


In [ ]:
# Demonstrate feature extraction concepts
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def spectral_entropy(signal, fs=1000, band=(12, 30)):
    """Compute spectral entropy in a frequency band."""
    from scipy.signal import welch
    f, psd = welch(signal, fs=fs, nperseg=min(len(signal), 256))
    band_mask = (f >= band[0]) & (f <= band[1])
    psd_band = psd[band_mask]
    psd_norm = psd_band / psd_band.sum()
    psd_norm = psd_norm[psd_norm > 0]
    return -np.sum(psd_norm * np.log(psd_norm))

# Simulate normal vs seizure EEG
np.random.seed(42)
fs = 1000
t = np.linspace(0, 5, fs * 5)

# Normal: low amplitude, complex
normal_eeg = np.random.randn(len(t)) * 20

# Seizure: rhythmic beta oscillation
seizure_eeg = (np.sin(2 * np.pi * 20 * t) * 80 +
               np.random.randn(len(t)) * 15)

features = {
    'Condition': ['Normal', 'Seizure'],
    'Variance': [np.var(normal_eeg), np.var(seizure_eeg)],
    'Skewness': [stats.skew(normal_eeg), stats.skew(seizure_eeg)],
    'Spectral Entropy': [spectral_entropy(normal_eeg), spectral_entropy(seizure_eeg)]
}

import pandas as pd
df = pd.DataFrame(features).set_index('Condition')
print("Feature comparison — Normal vs Seizure EEG:")
print(df.round(3))


---
## Step 4 — Connectivity Matrix Visualization
### `04_siencehisto.py`

**Purpose:** Generate bar chart visualizations from the wPLI connectivity Excel matrices, showing per-channel mean connectivity strength.

**Output:** One bar chart per file per frequency band, with a threshold line at 0.3 to highlight highly connected channels.


In [ ]:
# Demo: Channel connectivity bar chart
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
channels_16 = ['Fp1','Fp2','F7','F3','Fz','F4','F8',
               'T3','C3','Cz','C4','T4','T5','P3','Pz','P4']

# Simulate mean node strength per channel
node_strength = np.random.uniform(0.35, 0.75, 16)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(channels_16, node_strength, color='steelblue', edgecolor='navy', alpha=0.8)
ax.axhline(y=0.3, color='red', linestyle='--', linewidth=1.5, label='Threshold (0.3)')
ax.set_xlabel('Channel')
ax.set_ylabel('Mean wPLI Connectivity')
ax.set_title('Channel Connectivity Bar Chart — Beta Band (12–30 Hz)', fontsize=11)
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('demo_channel_bar.png', dpi=100, bbox_inches='tight')
plt.show()
print("All channels exceed threshold — strong seizure-period connectivity ✓")


---
## Step 5 — Source-Level Analysis with dSPM + HCPMMP1
### `05_dspm_connectivity.py`

**Purpose:** Project scalp EEG into source space using dynamic Statistical Parametric Mapping (dSPM), then parcellate into brain regions using the HCP Multi-Modal Parcellation (HCPMMP1) atlas.

### Pipeline:

```
Preprocessed EEG
      │
      ▼
BEM forward model (3-layer: brain/skull/scalp)
      │
      ▼
Minimum norm inverse solution (dSPM)
      │
      ▼
HCPMMP1 parcellation → 18 bilateral ROIs (9 networks)
      │
      ▼
wPLI source-space connectivity (θ / α / β)
      │
      ▼
GATv2 graph feature matrix
```

### 18 ROIs across 9 Networks:

| Network | ROIs |
|---------|------|
| Prefrontal | L/R PFC |
| Motor | L/R M1 |
| Somatosensory | L/R S1 |
| Temporal | L/R Superior Temporal |
| Parietal | L/R Inferior Parietal |
| Occipital | L/R V1 |
| Cingulate | L/R ACC |
| Insula | L/R Insula |
| Hippocampal | L/R Hippocampus |

> **Why source space?** Scalp EEG suffers from volume conduction — nearby channels are artificially correlated. Source projection reduces this, giving cleaner connectivity estimates.


In [ ]:
# Conceptual: Source-space connectivity graph structure
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 18 ROI labels (9 networks x 2 hemispheres)
roi_labels = [
    'L-PFC', 'R-PFC', 'L-M1', 'R-M1',
    'L-S1', 'R-S1', 'L-ST', 'R-ST',
    'L-IP', 'R-IP', 'L-V1', 'R-V1',
    'L-ACC', 'R-ACC', 'L-Ins', 'R-Ins',
    'L-Hipp', 'R-Hipp'
]
n_roi = len(roi_labels)

# Simulate source-space wPLI matrix
np.random.seed(42)
src_wpli = np.random.uniform(0.1, 0.6, (n_roi, n_roi))
np.fill_diagonal(src_wpli, 0)
src_wpli = (src_wpli + src_wpli.T) / 2

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(src_wpli, cmap='RdYlBu_r', vmin=0, vmax=0.7)
ax.set_xticks(range(n_roi))
ax.set_yticks(range(n_roi))
ax.set_xticklabels(roi_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(roi_labels, fontsize=8)
ax.set_title('Source-Space wPLI Connectivity (18 ROIs × 18 ROIs)
HCPMMP1 Parcellation — Beta Band', fontsize=10)
plt.colorbar(im, ax=ax, label='wPLI')
plt.tight_layout()
plt.savefig('demo_source_connectivity.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Graph input shape for GATv2: {n_roi} nodes × {n_roi} edges (per band)")
print(f"Total edge features (3 bands): {n_roi} × {n_roi} × 3 = {n_roi*n_roi*3}")


---
## Step 6 — GATv2 Seizure Classification
### `06_gatv2_classification.py`

**Purpose:** Apply a Graph Attention Network v2 (GATv2) to classify seizure vs. non-seizure brain states using the source-space connectivity graphs from Step 5.

### Model Architecture:

```
Input: 18-node graph (wPLI edge weights, 3 bands)
       │
       ▼
GATv2 Conv Layer 1 (64 heads × 8 features = 512 hidden)
       │
       ▼
GATv2 Conv Layer 2 (32 features)
       │
       ▼
Global Mean Pooling
       │
       ▼
Linear → Dropout(0.3) → Linear
       │
       ▼
Output: Seizure probability (binary classification)
```

### Training Protocol:
| Setting | Value |
|---------|-------|
| Validation | Leave-One-Subject-Out (LOSO) |
| Optimizer | Adam (lr=0.001) |
| Loss | Binary Cross-Entropy |
| Epochs | 100 |
| Random seed | 42 |

### Key Results (v4):
| Metric | Value |
|--------|-------|
| LOSO AUC | 1.00 |
| Permutation p-value | 0.000 (all bands) |
| Best band | Beta (12–30 Hz) |


In [ ]:
# GATv2 model architecture overview
import torch
import torch.nn as nn

class EpiConnectomeGATv2(nn.Module):
    """
    Simplified GATv2 architecture for EpiConnectome.
    In the full pipeline, torch_geometric.nn.GATv2Conv is used.
    """
    def __init__(self, in_channels=3, hidden=64, heads=8, num_classes=1):
        super().__init__()
        self.description = {
            'input': f'{in_channels} edge features (theta/alpha/beta wPLI)',
            'layer1': f'GATv2Conv({in_channels} → {hidden}, heads={heads})',
            'layer2': f'GATv2Conv({hidden*heads} → 32)',
            'pooling': 'Global Mean Pooling over 18 nodes',
            'output': f'Linear → Sigmoid → seizure probability'
        }

    def forward(self, x):
        return x  # placeholder

model = EpiConnectomeGATv2()
print("EpiConnectome GATv2 Architecture:")
print("-" * 45)
for k, v in model.description.items():
    print(f"  {k:10s}: {v}")
print()
print("Dataset summary:")
print("  Subjects  : 14 (LOSO cross-validation)")
print("  Seizures  : 47")
print("  Non-seizure: matched windows")
print("  Graph nodes: 18 ROIs (HCPMMP1)")
print("  Edge features: 3 (theta / alpha / beta wPLI)")
print()
print("Results:")
print("  LOSO AUC     : 1.00")
print("  Permutation p: 0.000 (all frequency bands)")


---
## 📊 Pipeline Summary

```
Siena EDF Files (14 patients, 47 seizures)
         │
         ▼  Step 1: 01_siena_to_txt_converter.py
    TXT Segments
         │
         ▼  Step 2: 02_channelbasedcode_Siena.py
    wPLI Connectivity Matrices (θ/α/β)
    + Graph Theory Metrics
    + Circle Plots + Bar Charts
         │
         ▼  Step 3: 03_siena_feature_extraction.py
    EEG Features (SampEn, SpecEn, Var, Skew)
         │
         ▼  Step 4: 04_siencehisto.py
    Connectivity Visualizations
         │
         ▼  Step 5: 05_dspm_connectivity.py
    Source-Space Graphs (18 ROIs, HCPMMP1)
         │
         ▼  Step 6: 06_gatv2_classification.py
    GATv2 Classification
    AUC = 1.00 | p = 0.000
```

## 📚 References

1. Detti, P. (2020). Siena Scalp EEG Database. PhysioNet. https://doi.org/10.13026/5d4a-j060
2. Gramfort, A. et al. (2013). MNE software for processing MEG and EEG data. NeuroImage.
3. Brody, S. et al. (2022). How Attentive are Graph Attention Networks? ICLR 2022.
4. Nolte, G. et al. (2008). Robustly estimating the flow direction of information. NeuroImage.

---
*EpiConnectome | Brainhack School 2026 | github.com/duhmariya/EEG_Project*
